In [6]:
import pandas as pd
import geopandas as gpd

In [7]:
pd.options.display.float_format = "{:,.2f}".format

In [8]:
p1 = pd.read_csv('./inputs/parcels_preprocessed.csv')

In [23]:
p1.columns

Index(['PID', 'CONDO_ID', 'NUM_BLDGS', 'LU', 'LU_DESC', 'BLDG_TYPE',
       'RES_FLOOR', 'RES_UNITS', 'TT_RMS', 'BED_RMS', 'FULL_BTH', 'HLF_BTH',
       'KITCHENS', 'OVERALL_COND', 'INT_COND', 'EXT_COND', 'NUM_PARKING',
       'STRUCTURE_CLASS', 'YR_REMODEL', 'YR_BUILT', 'LAND_VALUE', 'BLDG_VALUE',
       'TOTAL_VALUE', 'LAND_SF', 'GROSS_AREA', 'LIVING_AREA', 'zoning_use',
       'max_far', 'max_height', 'front_setback', 'side_setback',
       'rear_setback', 'max_dua', 'max_floors', 'neighborhood_name',
       'neighborhood_id', 'median_hh_income', 'emp_dist_m',
       'flag_multiple_2015_in_2025', 'lu_2015', 'land_sf_2015',
       'flag_multiple_2025_in_2015', 'geometry'],
      dtype='object')

In [28]:
p1.loc[(p1.LU != p1.lu_2015) & (p1.flag_multiple_2025_in_2015 == True), ["LU", "LAND_SF", "lu_2015", "land_sf_2015"]]

,LU,LAND_SF,lu_2015,land_sf_2015
446,EA,"58,991.00",E,"820,270.00"
2781,RC,"28,157.00",E,"37,542.00"
5732,R3,"6,037.00",R2,"8,705.00"
11523,R4,"5,729.00",R2,"8,705.00"
11689,R1,"4,130.00",E,"7,200.00"
...,...,...,...,...
98503,RL - RL,"10,055.00",RL,"83,550.00"
98504,RL - RL,"9,258.00",RL,"83,550.00"
98505,RL - RL,"9,047.00",RL,"83,550.00"
98506,RL - RL,"3,558.00",RL,"83,550.00"


collapse 2015 parcels

In [3]:
p1 = pd.read_csv('./preprocessing/raw_data/boston_parcel_assessors_2015.csv')
ps1 = gpd.read_file('./preprocessing/raw_data/boston_parcel_shapes_2015.geojson')

In [4]:
# collapse condos records to they have one physical space- they have multiple PID per CM_ID
# aggregate appropriate attributes and assign to one PID
source = p1[["CM_ID", "PID", "LU", "LAND_SF"]]
mask_cm = source["CM_ID"].notna()

p1_no_cm = source.loc[~mask_cm, ["CM_ID", "PID", "LU", "LAND_SF"]]

p1_cm = (
    source.loc[mask_cm, ["CM_ID", "PID", "LU", "LAND_SF"]]
    .assign(anchor_lu=lambda d: d["LU"].where(d["PID"].eq(d["CM_ID"])))
    .groupby("CM_ID", as_index=False)
    .agg(
        LAND_SF=("LAND_SF", "sum"),
        LU=("anchor_lu", "first")
    )
    .assign(PID=lambda d: d["CM_ID"])
    [["CM_ID", "PID", "LU", "LAND_SF"]]
)

p1 = pd.concat([p1_no_cm, p1_cm], ignore_index=True)

In [5]:
# collapse condo parcels so they are not stacked- they have multiple parcels per GIS_ID
ps1 = ps1.drop_duplicates(subset="TaxData2015_GIS_ID", keep="first").copy()
ps1 = ps1.reset_index(drop=True)

In [6]:
# merge parcels and their shapes
p1.PID = p1.PID.str.strip('_').astype(int)
ps1.TaxData2015_GIS_ID = ps1.TaxData2015_GIS_ID.astype(int)

p1s = p1.merge(ps1, left_on='PID', right_on='TaxData2015_GIS_ID', how='left')

In [7]:
# we still have some parcels that don't have a shape, but solved the main problem
len(p1s), len(p1s[p1s.geometry.isnull()])

(103215, 2007)

In [8]:
# they're mostly condo parking, which shouldn't be redeveloped anyway
p1s[p1s.geometry.isnull()].LU.value_counts().head()

LU
CP    798
E     341
CC    256
C     241
RC    133
Name: count, dtype: int64

repeat for 2025 parcels

In [9]:
p2 = pd.read_csv('./preprocessing/raw_data/boston_parcel_assessors_2025.csv')
ps2 = gpd.read_file('./preprocessing/raw_data/boston_parcel_shapes_2025.geojson')

C:\Users\etheo\AppData\Local\Temp\ipykernel_45240\2490641573.py:1: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  p2 = pd.read_csv('./preprocessing/raw_data/boston_parcel_assessors_2025.csv')


In [10]:
source = p2[["CM_ID", "PID", "LU", "LAND_SF"]]
mask_cm = source["CM_ID"].notna()

p2_no_cm = source.loc[~mask_cm, ["CM_ID", "PID", "LU", "LAND_SF"]]

p2_cm = (
    source.loc[mask_cm, ["CM_ID", "PID", "LU", "LAND_SF"]]
    .assign(anchor_lu=lambda d: d["LU"].where(d["PID"].eq(d["CM_ID"])))
    .groupby("CM_ID", as_index=False)
    .agg(
        LAND_SF=("LAND_SF", "sum"),
        LU=("anchor_lu", "first")
    )
    .assign(PID=lambda d: d["CM_ID"])
    [["CM_ID", "PID", "LU", "LAND_SF"]]
)

p2 = pd.concat([p2_no_cm, p2_cm], ignore_index=True)

In [11]:
ps2 = ps2[
    ~ps2["MAP_PAR_ID"].astype(str).str.strip().isin(["MASSGIS", "ISLAND"])
].copy()

ps2 = ps2.reset_index(drop=True)

In [12]:
ps2.MAP_PAR_ID = ps2.MAP_PAR_ID.astype(int)

p2s = p2.merge(ps2, left_on='PID', right_on='MAP_PAR_ID', how='left')

In [13]:
len(p2s), len(p2s[p1s.geometry.isnull()])

C:\Users\etheo\AppData\Local\Temp\ipykernel_45240\23006613.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  len(p2s), len(p2s[p1s.geometry.isnull()])


(99689, 2003)

In [ ]:
p2s[p2s.geometry.isnull()].LU.value_counts().head()

LU
E     576
C     269
A      89
I      67
CM     62
Name: count, dtype: int64

spatially join the two datasets

In [ ]:
p1s = gpd.GeoDataFrame(p1s, geometry='geometry', crs=ps1.crs)
p2s = gpd.GeoDataFrame(p2s, geometry='geometry', crs=ps2.crs)


In [ ]:
projected_crs = "EPSG:26986"

if p1s.crs is None:
    p1s = p1s.set_crs("EPSG:4326")
if p2s.crs is None:
    p2s = p2s.set_crs("EPSG:4326")

p1s = p1s.to_crs(projected_crs)
p2s = p2s.to_crs(projected_crs)

In [ ]:
from shapely import make_valid


def clean_geom(geom):
    if geom is None or geom.is_empty:
        return None
    try:
        fixed = make_valid(geom) if not geom.is_valid else geom
    except Exception:
        fixed = geom
    if fixed is None or fixed.is_empty:
        return None
    if not fixed.is_valid:
        try:
            fixed = fixed.buffer(0)
        except Exception:
            return None
    return fixed if fixed is not None and not fixed.is_empty else None


def safe_intersection_area(left_geom, right_geom):
    left_fixed = clean_geom(left_geom)
    right_fixed = clean_geom(right_geom)
    if left_fixed is None or right_fixed is None:
        return 0.0
    try:
        return left_fixed.intersection(right_fixed).area
    except Exception:
        try:
            return left_fixed.buffer(0).intersection(right_fixed.buffer(0)).area
        except Exception:
            return 0.0


# 1) Keep only needed fields and create stable row ids
p1_base = (
    p1s[["LU", "LAND_SF", "geometry"]]
    .copy()
    .reset_index()
    .rename(columns={"index": "p1_row", "LU": "p1_LU", "LAND_SF": "p1_LAND_SF"})
)

p2_base = p2s.copy().reset_index().rename(columns={"index": "p2_row"})

# Repair invalid geometries before spatial operations.
p1_base["geometry"] = p1_base["geometry"].apply(clean_geom)
p2_base["geometry"] = p2_base["geometry"].apply(clean_geom)

p1_base = p1_base[p1_base.geometry.notna() & ~p1_base.geometry.is_empty].copy()
p2_base = p2_base[p2_base.geometry.notna() & ~p2_base.geometry.is_empty].copy()

# 2) Build all p2<->p1 spatial matches
pairs = gpd.sjoin(
    p2_base[["p2_row", "geometry"]],
    p1_base[["p1_row", "p1_LU", "p1_LAND_SF", "geometry"]],
    how="left",
    predicate="intersects",
)

pairs_matched = pairs[pairs["p1_row"].notna()].copy()

# Defaults in case nothing matches
p2_with_p1 = p2_base.copy()
p2_with_p1["p1_row"] = pd.NA
p2_with_p1["p1_LU"] = pd.NA
p2_with_p1["p1_LAND_SF"] = pd.NA
p2_with_p1["flag_multiple_p1_in_p2"] = False
p2_with_p1["flag_multiple_p2_in_p1"] = False

if not pairs_matched.empty:
    # 3) Compute overlap size to choose "largest" match when multiple p1 are in one p2
    pairs_matched["p1_row"] = pairs_matched["p1_row"].astype(int)
    pairs_matched = pairs_matched.merge(
        p1_base[["p1_row", "geometry"]].rename(columns={"geometry": "p1_geometry"}),
        on="p1_row",
        how="left",
    )
    pairs_matched["intersection_area"] = pairs_matched.apply(
        lambda row: safe_intersection_area(row["geometry"], row["p1_geometry"]),
        axis=1,
    )
    pairs_matched["p1_shape_area"] = pairs_matched["p1_geometry"].apply(
        lambda geom: geom.area if geom is not None else 0.0
    )

    # Flag p2 rows that intersect multiple p1 rows
    flag_multi_p1 = (
        pairs_matched.groupby("p2_row")["p1_row"]
        .nunique()
        .gt(1)
        .rename("flag_multiple_p1_in_p2")
        .reset_index()
    )

    # 4) For each p2, keep the p1 with largest overlap (tie-breaker: larger p1 area)
    best_for_p2 = (
        pairs_matched.sort_values(
            ["p2_row", "intersection_area", "p1_shape_area"],
            ascending=[True, False, False],
        )
        .drop_duplicates(subset="p2_row", keep="first")
        .copy()
    )

    # 5) Flag p1 rows that were assigned to multiple p2 rows
    multi_p2_ids = (
        best_for_p2.groupby("p1_row")["p2_row"].nunique().loc[lambda s: s > 1].index
    )
    best_for_p2["flag_multiple_p2_in_p1"] = best_for_p2["p1_row"].isin(multi_p2_ids)

    # If one p1 is in multiple p2, keep p1 attrs in only one p2 row (largest overlap)
    best_for_p2["rank_within_p1"] = best_for_p2.groupby("p1_row")[
        "intersection_area"
    ].rank(method="first", ascending=False)
    loser_mask = best_for_p2["flag_multiple_p2_in_p1"] & (best_for_p2["rank_within_p1"] > 1)
    best_for_p2.loc[loser_mask, ["p1_row", "p1_LU", "p1_LAND_SF"]] = pd.NA

    # 6) Attach chosen p1 attributes + flags back to all p2 rows
    p2_with_p1 = p2_base.merge(flag_multi_p1, on="p2_row", how="left")
    p2_with_p1 = p2_with_p1.merge(
        best_for_p2[["p2_row", "p1_row", "p1_LU", "p1_LAND_SF", "flag_multiple_p2_in_p1"]],
        on="p2_row",
        how="left",
    )

    p2_with_p1["flag_multiple_p1_in_p2"] = p2_with_p1["flag_multiple_p1_in_p2"].fillna(False).astype(bool)
    p2_with_p1["flag_multiple_p2_in_p1"] = p2_with_p1["flag_multiple_p2_in_p1"].fillna(False).astype(bool)

AttributeError: 'Series' object has no attribute 'is_empty'

In [ ]:
p2_new = p2_with_p1[['LU', 'LAND_SF', 'p1_row', 'p1_LAND_SF', 
                    'p1_LU', 'flag_multiple_p1_in_p2', 'flag_multiple_p2_in_p1', "geometry"]]

In [ ]:
p2_new[p2_new.LU != p2_new.p1_LU].head(3)

In [ ]:
len(p2_new[p2_new.LU != p2_new.p1_LU])